# Validation of the Mosquito Suitability Model

This notebook documents the validation workflow used to compare the mosquito suitability model against occurrence records from Kraemer et al. (2015) It is included as a technical appendix to support transparency and reproducibility.

**Approach:**
1. Aggregate monthly suitability scores to city-level metrics (peak month, season length, mean active-month score)
2. Spatial join: for each city, check whether any Kraemer occurrence falls within a given radius
3. Compare score distributions between presence and absence groups
4. Quantify discrimination with ROC/AUC

**Data:**
- `mosquito_suitability.csv`: 1,421 cities × 12 months, ERA5 1991–2020 climate normals
- `kraemer_occurrences.csv`: 42,066 occurrence records, 1958–2014, both species

**Key limitation:** Kraemer records go up to 2014; suitability scores are based on 1991–2020 climate normals. This temporal mismatch is unavoidable but acceptable: climate normals are stable enough over this window, and the comparison is between long-run climatic suitability and observed establishment, not short-term forecasting.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.spatial import cKDTree
from sklearn.metrics import roc_curve, auc, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

RADIUS_KM = 50  # matching radius for spatial join
EARTH_RADIUS_KM = 6371.0

print('Libraries loaded.')

## 1. Load Data

Both input files are the outputs of a separate harmonization pipeline (`00_harmonize.py`), run in Google Colab prior to this analysis.

For `kraemer_occurrences.csv`, the harmonization steps were:
- Column renaming: `VECTOR` → `species`, `Y`/`X` → `lat`/`lon`, `COUNTRY` → `country_name`, `COUNTRY_ID` → `iso3`
- Species label standardization: `'Aedes aegypti'` → `'Ae. aegypti'`, `'Aedes albopictus'` → `'Ae. albopictus'`
- Addition of `is_point` flag: `True` where `location_type == 'point'`, `False` for polygon centroids
- Rows without coordinates dropped; `year` cast to nullable integer

The `is_point` flag is used in Section 3 to restrict the spatial join to point-level records only, excluding polygon centroids that may represent regions up to hundreds of km wide.

In [ ]:
suit = pd.read_csv('mosquito_suitability.csv')
kraemer = pd.read_csv('01_kraemer_occurrences.csv')

print(f'Suitability rows:  {len(suit):,}  ({suit["city"].nunique()} cities)')
print(f'Kraemer records:   {len(kraemer):,}')
print()
print('Kraemer species breakdown:')
print(kraemer['species'].value_counts())
print()
print('Kraemer location_type breakdown:')
print(kraemer['location_type'].value_counts())

## 2. Aggregate Suitability to City Level

From 12 monthly rows per city, derive city-level metrics per species:
- **`peak_score`**: maximum monthly suitability (best-month performance)
- **`season_months_02/03/04`**: number of months exceeding a suitability threshold of 0.2, 0.3, or 0.4 respectively. A threshold of `> 0` was deliberately avoided: for a multiplicative score, values just above zero can reflect negligible climatic suitability while still being counted as active months, which distorts season length. The three thresholds correspond directly to the filter options available in the Tableau dashboard, keeping the validation consistent with the published tool. Multiple plausible thresholds are tested to confirm that the result does not depend on any single threshold choice.
- **`mean_active_score`**: mean suitability across months with score >= 0.2 (the lowest dashboard threshold)

In [ ]:
def city_metrics(df, species_col):
    """Aggregate monthly scores to city-level metrics."""
    grp = df.groupby(['city', 'country', 'lat', 'lon'])
    peak = grp[species_col].max().rename('peak_score')

    # Season length at three thresholds matching the Tableau dashboard filters
    season_02 = (df[species_col] >= 0.2).groupby(
        [df['city'], df['country'], df['lat'], df['lon']]
    ).sum().rename('season_months_02')
    season_03 = (df[species_col] >= 0.3).groupby(
        [df['city'], df['country'], df['lat'], df['lon']]
    ).sum().rename('season_months_03')
    season_04 = (df[species_col] >= 0.4).groupby(
        [df['city'], df['country'], df['lat'], df['lon']]
    ).sum().rename('season_months_04')

    # Mean score across months that clear the lowest threshold (0.2)
    mean_active = df[df[species_col] >= 0.2].groupby(
        ['city', 'country', 'lat', 'lon']
    )[species_col].mean().rename('mean_active_score')

    result = pd.concat([peak, season_02, season_03, season_04, mean_active], axis=1).reset_index()
    result['mean_active_score'] = result['mean_active_score'].fillna(0)
    return result

cities_aeg = city_metrics(suit, 'suitability_score_aegypti')
cities_alb = city_metrics(suit, 'suitability_score_albopictus')

print('Ae. aegypti city metrics:')
print(cities_aeg[['peak_score', 'season_months_02', 'season_months_03', 'season_months_04', 'mean_active_score']].describe().round(3))
print()
print('Ae. albopictus city metrics:')
print(cities_alb[['peak_score', 'season_months_02', 'season_months_03', 'season_months_04', 'mean_active_score']].describe().round(3))

## 3. Spatial Join: Assign Presence/Absence Labels

For each city, check whether any Kraemer occurrence record falls within `RADIUS_KM`.

**Design choices:**
- Only `is_point == True` records are used. Polygon centroids can represent regions hundreds of km wide, which would introduce false positives at city level.
- Both species are handled separately.
- The join is done with a 3D unit-sphere kd-tree (fast, accurate for any lat/lon range).

In [ ]:
def latlon_to_xyz(lat_deg, lon_deg):
    """Convert lat/lon (degrees) to unit-sphere XYZ."""
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    return np.column_stack([x, y, z])

def spatial_presence_labels(city_df, occ_df, radius_km=50):
    """
    For each city in city_df, assign presence=1 if any occurrence in occ_df
    falls within radius_km. Uses cKDTree on unit sphere.
    """
    # chord length threshold on unit sphere
    chord_threshold = 2 * np.sin(np.radians(radius_km / EARTH_RADIUS_KM / 2))
    
    city_xyz = latlon_to_xyz(city_df['lat'].values, city_df['lon'].values)
    occ_xyz  = latlon_to_xyz(occ_df['lat'].values, occ_df['lon'].values)
    
    tree = cKDTree(occ_xyz)
    # query: for each city, find nearest occurrence
    dist, _ = tree.query(city_xyz, k=1)
    
    presence = (dist <= chord_threshold).astype(int)
    return presence

# Filter to point-only records per species
k_aeg = kraemer[(kraemer['species'] == 'Ae. aegypti') & (kraemer['is_point'] == True)].dropna(subset=['lat','lon'])
k_alb = kraemer[(kraemer['species'] == 'Ae. albopictus') & (kraemer['is_point'] == True)].dropna(subset=['lat','lon'])

print(f'Kraemer point records used — Ae. aegypti: {len(k_aeg):,} | Ae. albopictus: {len(k_alb):,}')

cities_aeg['presence'] = spatial_presence_labels(cities_aeg, k_aeg, radius_km=RADIUS_KM)
cities_alb['presence'] = spatial_presence_labels(cities_alb, k_alb, radius_km=RADIUS_KM)

for label, df in [('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]:
    n1 = df['presence'].sum()
    n0 = len(df) - n1
    print(f'{label}: {n1} presence cities, {n0} absence cities  ({100*n1/len(df):.1f}% presence rate)')

## 4. Sensitivity Analysis: Radius Choice

The 50 km radius is a modelling decision. The table below shows how both presence rate and AUC (for `season_months_02`, the strongest discriminator) change across radii. Stable AUC across radii indicates that the main conclusions do not hinge on the choice of 50 km.

In [ ]:
radii = [25, 50, 75, 100, 150]
results = []
for r in radii:
    for label, city_df, occ_df in [('Ae. aegypti', cities_aeg, k_aeg), ('Ae. albopictus', cities_alb, k_alb)]:
        pres_labels = spatial_presence_labels(city_df, occ_df, radius_km=r)
        n_pres = pres_labels.sum()
        # AUC for season_months_02 at this radius
        tmp = city_df.copy()
        tmp['presence'] = pres_labels
        if tmp['presence'].sum() > 0:
            fpr, tpr, _ = roc_curve(tmp['presence'], tmp['season_months_02'])
            roc_auc = round(auc(fpr, tpr), 3)
        else:
            roc_auc = None
        results.append({'radius_km': r, 'species': label,
                        'n_presence': n_pres,
                        'pct_presence': round(100*n_pres/len(city_df), 1),
                        'AUC_season_02': roc_auc})

print(pd.DataFrame(results).to_string(index=False))

## 5. Distribution Comparison: Presence vs. Absence-Labelled Cities

Visualise and test whether suitability scores differ between presence-labelled and absence-labelled cities. Note that absence-labelled here means no Kraemer point record within 50 km, not confirmed biological absence. The negative class is noisy and should be interpreted accordingly.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Suitability Score Distribution: Presence vs. Absence\n(Kraemer et al. 2015/2017, point records only, 50 km radius)',
             fontsize=13, y=1.01)

metrics = ['peak_score', 'season_months', 'mean_active_score']
xlabels = ['Peak monthly suitability', 'Season length (months > 0)', 'Mean score (active months)']
colors = {'presence': '#e05c5c', 'absence': '#5b8ed6'}

for row, (species_label, df) in enumerate([('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]):
    pres = df[df['presence'] == 1]
    abs_ = df[df['presence'] == 0]
    
    for col, (metric, xlabel) in enumerate(zip(metrics, xlabels)):
        ax = axes[row, col]
        
        ax.hist(abs_[metric], bins=30, alpha=0.6, color=colors['absence'],
                density=True, label=f'Absence (n={len(abs_)})')
        ax.hist(pres[metric], bins=30, alpha=0.6, color=colors['presence'],
                density=True, label=f'Presence (n={len(pres)})')
        
        # Mann-Whitney U test
        stat, p = stats.mannwhitneyu(pres[metric], abs_[metric], alternative='greater')
        p_str = f'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
        ax.set_title(f'{species_label}\n{xlabel}\nMann-Whitney: {p_str}', fontsize=9)
        ax.legend(fontsize=8)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel('Density', fontsize=8)

plt.tight_layout()
plt.savefig('fig_01_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_01_score_distributions.png')

## 6. Boxplots with Effect Size (Cohen's d)

In [ ]:
def cohens_d(a, b):
    """Pooled Cohen's d for two independent samples."""
    pooled_std = np.sqrt(((len(a)-1)*a.std()**2 + (len(b)-1)*b.std()**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / pooled_std if pooled_std > 0 else 0

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Peak Monthly Suitability: Presence vs. Absence Cities', fontsize=12)

for ax, (species_label, df) in zip(axes, [('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]):
    pres = df[df['presence'] == 1]['peak_score']
    abs_ = df[df['presence'] == 0]['peak_score']
    
    data_bp = pd.DataFrame({
        'peak_score': pd.concat([pres, abs_]),
        'group': ['Presence']*len(pres) + ['Absence']*len(abs_)
    })
    
    sns.boxplot(data=data_bp, x='group', y='peak_score', ax=ax,
                palette={'Presence': '#e05c5c', 'Absence': '#5b8ed6'},
                order=['Presence', 'Absence'], width=0.5)
    sns.stripplot(data=data_bp, x='group', y='peak_score', ax=ax,
                  palette={'Presence': '#c03030', 'Absence': '#3a6ab0'},
                  order=['Presence', 'Absence'], alpha=0.15, size=2, jitter=True)
    
    stat, p = stats.mannwhitneyu(pres, abs_, alternative='greater')
    d = cohens_d(pres, abs_)
    p_str = 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
    
    ax.set_title(f'{species_label}\nMann-Whitney: {p_str} | Cohen\'s d = {d:.2f}', fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('Peak monthly suitability score')
    
    # Annotate medians
    for i, (group, values) in enumerate([('Presence', pres), ('Absence', abs_)]):
        med = values.median()
        ax.text(i, med + 0.02, f'median={med:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_02_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_02_boxplots.png')

## 7. ROC / AUC Analysis

AUC measures how well the suitability score discriminates between presence and absence cities. AUC = 0.5 is random; AUC = 1.0 is perfect. An AUC > 0.70 is generally considered acceptable discrimination in many applied classification settings, though this threshold is context-dependent. Ecological niche models typically produce lower AUC values than models with clean, confirmed absence data, partly because absence records in sparse global datasets often reflect surveillance gaps rather than confirmed biological absence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('ROC Curves: Suitability Score as Classifier of Presence/Absence', fontsize=12)

for ax, (species_label, df) in zip(axes, [('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]):
    metrics_roc = [
        ('peak_score',        'Peak score',              '#e05c5c'),
        ('season_months_02',  'Season length (≥ 0.2)',   '#5b8ed6'),
        ('season_months_03',  'Season length (≥ 0.3)',   '#3a6ab0'),
        ('season_months_04',  'Season length (≥ 0.4)',   '#1e3f6e'),
        ('mean_active_score', 'Mean active score',       '#5cad6e'),
    ]
    
    for metric, label, color in metrics_roc:
        fpr, tpr, _ = roc_curve(df['presence'], df[metric])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.set_title(species_label, fontsize=10)
    ax.legend(fontsize=7)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('fig_03_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_03_roc_curves.png')

## 8. Summary Statistics Table

In [ ]:
rows = []
for species_label, df in [('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]:
    pres = df[df['presence'] == 1]
    abs_ = df[df['presence'] == 0]
    
    for metric in ['peak_score', 'season_months_02', 'season_months_03', 'season_months_04', 'mean_active_score']:
        stat, p = stats.mannwhitneyu(pres[metric], abs_[metric], alternative='greater')
        d = cohens_d(pres[metric], abs_[metric])
        fpr, tpr, _ = roc_curve(df['presence'], df[metric])
        roc_auc = auc(fpr, tpr)
        
        rows.append({
            'Species': species_label,
            'Metric': metric,
            'Presence n': len(pres),
            'Absence n': len(abs_),
            'Presence median': round(pres[metric].median(), 3),
            'Absence median': round(abs_[metric].median(), 3),
            'MW p-value': '< 0.001' if p < 0.001 else round(p, 4),
            "Cohen's d": round(d, 3),
            'AUC': round(roc_auc, 3)
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv('validation_summary.csv', index=False)
print('\nSaved: validation_summary.csv')

## 9. Geographic Sanity Check

Plot presence/absence cities on a world map to verify the spatial join looks correct.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
fig.suptitle('Geographic Distribution of Presence/Absence Cities\n(50 km radius from Kraemer point records)',
             fontsize=12)

for ax, (species_label, df) in zip(axes, [('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]):
    pres = df[df['presence'] == 1]
    abs_ = df[df['presence'] == 0]
    
    ax.scatter(abs_['lon'], abs_['lat'], s=8, alpha=0.4, color='#5b8ed6', label=f'Absence (n={len(abs_)})', zorder=2)
    ax.scatter(pres['lon'], pres['lat'], s=15, alpha=0.7, color='#e05c5c', label=f'Presence (n={len(pres)})', zorder=3)
    
    ax.set_title(species_label, fontsize=10)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 75)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.axhline(0, color='grey', lw=0.5, alpha=0.5)
    ax.legend(fontsize=9, loc='lower left')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('fig_04_geographic_sanity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_04_geographic_sanity.png')

## 10. False Negatives: High-Suitability Cities with No Kraemer Record

Cities labelled as absence here are more accurately understood as cities with no record in the validation dataset, not confirmed absence. Absence in Kraemer reflects coverage limitations of the validation layer — particularly in Southeast Asia and Africa, where surveillance intensity was uneven through 2014. These high-suitability cities are worth flagging as analytically interesting precisely because the model predicts suitability that the validation data cannot confirm or deny.

In [ ]:
for species_label, df in [('Ae. aegypti', cities_aeg), ('Ae. albopictus', cities_alb)]:
    # High suitability but no Kraemer record within 50 km
    fn = df[(df['presence'] == 0) & (df['peak_score'] >= 0.6)].copy()
    fn = fn.sort_values('peak_score', ascending=False)
    print(f'\n{species_label} — high-suitability absence cities (peak >= 0.6): {len(fn)}')
    cols = ['city', 'country', 'lat', 'lon', 'peak_score', 'season_months_02', 'season_months_03', 'season_months_04']
    print(fn[cols].head(15).to_string(index=False))

## 11. Interpretation Notes

**What the results show**

In this validation, season length is the strongest discriminator across all thresholds tested. For *Ae. aegypti*, AUC reaches 0.834 (season_months_02); for *Ae. albopictus*, 0.749. Both are comfortably above 0.70, a threshold often treated as acceptable discrimination in applied settings, though such thresholds are context-dependent. Ecological niche models generally produce lower AUC values than models with clean, confirmed absence data, in part because absence labels in sparse global datasets often reflect surveillance gaps rather than confirmed biological absence. Presence-labelled cities show a median of 12 active months (at ≥ 0.2 threshold) compared with 6 months for absence-labelled cities, indicating a clear and interpretable separation.

`mean_active_score` summarises the mean suitability across months that clear the 0.2 threshold only — it captures the quality of the active season, not a year-round average. Cities with short seasons but high within-season scores will differ from cities with long but moderate seasons, and both patterns are ecologically meaningful.

Peak score alone is weaker (AUC ~0.66–0.69), which is substantively plausible. Many subtropical cities without a Kraemer record still reach high peak suitability values, because absence in the validation layer often reflects incomplete surveillance coverage rather than confirmed biological absence.

**Important: this is a relative validation, not an absolute one**

These AUC values measure how well the suitability metrics separate presence-labelled from absence-labelled cities. They do not test against confirmed absences. The negative class is noisy — it mixes true negatives, unsampled locations, and places omitted from Kraemer due to geographic unevenness. Results should be interpreted as discrimination against a noisy background, not as definitive ecological validation.

**Key limitation: sampling bias in the validation data**

Presence rates are low: 71 presence cities (5.0%) for *Ae. aegypti* and 23 (1.6%) for *Ae. albopictus*. This mainly reflects coverage limitations of the Kraemer validation layer rather than a failure of the suitability model itself. The dataset compares 1,421 cities against a relatively limited and geographically uneven global record base up to 2014. Many cities labelled as absence — particularly in Southeast Asia and Africa — are more accurately understood as cities with no record in the validation dataset, not confirmed absence. This sampling bias in the validation data is a central caveat in interpreting all results.

**High-suitability absence-labelled cities** (Section 10) should be interpreted as either genuine undetected presences, or true absences despite suitable climate due to local factors such as elevation, land use, or lack of breeding habitat. This ambiguity is inherent to any presence-only validation framework.